In [3]:
import os
import uuid
import hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from cryptography.fernet import Fernet
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

PATH = "complications.csv"


# Przygotowanie danych

df = pd.read_csv(PATH)

# Wybieramy kluczowe kolumny dla uproszczenia (dane demograficzne, kliniczne i target)
cols = ['ID', 'AGE', 'SEX', 'S_AD_ORIT', 'D_AD_ORIT', 'K_BLOOD', 'L_BLOOD', 'LET_IS']
df = df[cols].copy()

# Imputacja braków danych (medianą)
for col in ['AGE', 'S_AD_ORIT', 'D_AD_ORIT', 'K_BLOOD', 'L_BLOOD']:
    df[col] = df[col].fillna(df[col].median())

# Cel: 0 - przeżycie, 1 - zgon (LET_IS > 0)
df['LET_IS'] = (df['LET_IS'].fillna(0).astype(int) > 0).astype(int)
df["diagnosis"] = df["LET_IS"].map({1: "Lethal", 0: "Alive"})


# Szyfrowanie i integralność danych

key = Fernet.generate_key()
cipher = Fernet(key)
print("Klucz szyfrujący wygenerowany.")

# Szyfrowanie kolumny diagnozy
df["diagnosis_enc"] = df["diagnosis"].apply(
    lambda x: cipher.encrypt(x.encode("utf-8")).decode("utf-8")
)
print("\nPrzykład zaszyfrowanej diagnozy:")
print(df[["diagnosis", "diagnosis_enc"]].head())

# Funkcja do haszowania pliku
def compute_file_hash(path: str, algo: str = "sha256") -> str:
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

# Zapisanie wyczyszczonej wersji do nowego pliku
SECURE_PATH = "complications_secure.csv"
df.to_csv(SECURE_PATH, index=False)
original_hash = compute_file_hash(SECURE_PATH, "sha256")
print(f"\nSHA-256 pliku danych: {original_hash}")

Klucz szyfrujący wygenerowany.

Przykład zaszyfrowanej diagnozy:
  diagnosis                                      diagnosis_enc
0     Alive  gAAAAABp5hdDDWDvRNa92aQ38NLtW2hFVF6QxBWWenLdOx...
1     Alive  gAAAAABp5hdDy8ZOUm4zmWTN83z-cN7iu82tzxHNAY7us0...
2     Alive  gAAAAABp5hdDBSqsYEqHyTGn4GLO3IInR_65PY3iiXV774...
3     Alive  gAAAAABp5hdDFjFxDb3MeDb-LT1r3jitLQugxeKvz7wyex...
4     Alive  gAAAAABp5hdDgwJSRoPVdfghR9rQNrQlRbn6GwMGy273Ky...

SHA-256 pliku danych: 9f1faacfac09683e7f3553c1038b4582b164d8bab6cc90a30c78f265a5e85b8d
